In [ ]:

import matplotlib.pyplot as plt
import nibabel as nib
import pandas as pd
import numpy as np
import scipy.io 
import mne
import os
import sys
sys.path.insert(0, 'E:/workspace/my_py_toolbox/') 
from hm_tools import *

epoch_path = "E:/workspace/study2_escape_task_eeg/eeg_process/" # 分析哪个roi就输入哪个文件夹的数据

behavior_path = "E:/workspace/study2_escape_task_eeg/behavior/" # 分析哪个roi就输入哪个文件夹的数据

subjects = [4, 5, 6, 8, 9, 11, 12, 14, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38,39,40,41,42,43,45,46,47, 49,50,51,52] 



### 要先确保eeg数据epoch的顺序和behavior的顺序是按序列一一对应的

In [3]:
channel_location_dict = {"Fp1":	[-26.25, 83, -18.25],
"Fp2": [26.25, 83, -18.25],
"F3": [-48.75, 57.5, 36.5],
"F4": [48.75, 57.5, 36.5],
"C3": [-63.75, -13, 65.75],
"C4": [63.75, -13, 65.75],
"P3": [-48, -86.5, 53],
"P4": [48, -86.5, 53],
"O1": [-27, -118, 4.25],
"O2": [27, -118, 4.25],
"F7": [-69, 44, -18.25],
"F8": [69, 44, -18.25],
"T7": [-84, -18.25, -13],
"T8": [84, -18.25, -13],
"P7": [-69, -80.5, -4.75],
"P8": [69, -80.5, -4.75],
"Fz": [0, 63.5, 59],
"Cz": [0, -10, 99.5],
"Pz": [0, -88, 77],
"FC1": [-32.25, 28.25, 77.75],
"FC2": [32.25, 28.25, 77.75],
"CP1": [-35.25, -52, 90.5],
"CP2": [35.25, -52, 90.5],
"FC5": [-75.75, 20, 21.5],
"FC6": [75.75, 20, 21.5],
"CP5": [-78, -51.25, 31.25],
"CP6": [78, -51.25, 31.25],
"FT9": [-72.75, 8.75, -55.75],
"FT10": [72.75, 8.75, -55.75],
"TP9": [-71.25, -49, -49.75],
"TP10": [71.25, -49, -49.75],
"F1": [-25.5, 62, 53.75],
"F2": [25.5, 62, 53.75],
"C1": [-35.25, -11.5, 89.75],
"C2": [35.25, -11.5, 89.75],
"P1": [-27, -88, 70.25],
"P2": [27, -88, 70.25],
"AF3": [-32.25, 80.75, 11.75],
"AF4": [32.25, 80.75, 11.75],
"FC3": [-59.25, 25.25, 54.5],
"FC4": [59.25, 25.25, 54.5],
"CP3": [-63, -52, 66.5],
"CP4": [63, -52, 66.5],
"PO3": [-31.5, -109.75, 32.75],
"PO4": [31.5, -109.75, 32.75],
"F5": [-63, 51.5, 11.75],
"F6": [63, 51.5, 11.75],
"C5": [-81.75, -14.5, 29],
"C6": [81.75, -14.5, 29],
"P5": [-63.75, -83.5, 26.75],
"P6": [63.75, -83.5, 26.75],
"AF7": [-50.25, 68, -19],
"AF8": [50.25, 68, -19],
"FT7": [-79.5, 15.5, -16],
"FT8": [79.5, 15.5, -16],
"TP7": [-80.25, -49.75, -8.5],
"TP8": [80.25, -49.75, -8.5],
"PO7": [-50.25, -103, 0.5],
"PO8": [50.25, -103, 0.5],
"Fpz": [0, 83, -18.25],
"CPz": [0, -53.5, 98],
"POz": [0, -109, 44],
"Oz": [0, -122.5, 8]}
# ['Fp1', 'Fp2', 'Fz', 'Cz', 'Pz', 'Fpz', 'CPz', 'POz', 'Oz'].
mymontage = mne.channels.make_dig_montage(ch_pos=channel_location_dict)#, nasion=[0, -10, 99.5], lpa=[-71.25, -49, -49.75], 
#                                        rpa=[71.25, -49, -49.75])

In [4]:
def gaussian_filter(data, time_span=50, fs=1000, axis=-1):

    # time_span: the length of the window (ms)
    from scipy import signal, ndimage

    winsize = time_span * fs / 1000

    # create a window(kernel) with value follow a gaussian distrubution
    window = signal.gaussian(winsize,std = int((winsize)/5) ) # Note! In matlab function 'gausswin',the parameter is α(defaut = 2.5), here is std, std = winsize/2α

    # Make the sum of the window = 1
    gusWin = window/sum(window)

    # convolve the last dimension
    data_filtered = ndimage.convolve1d(input=data, weights=gusWin, axis=axis, mode='nearest', origin=0)

    return data_filtered

In [ ]:
#个体水平
from mne.time_frequency import tfr_morlet


freqs=np.arange(4, 31, 1)
n_cycles = np.linspace(2, 5, num=27)
freqs = freqs[0:4]
n_cycles = n_cycles[0:4]
# freqs = freqs[4:9]
# n_cycles = n_cycles[4:9]
# freqs = freqs[9:]
# n_cycles = n_cycles[9:]

# n_cycles =4
time_span = 200
ch_names = []
decision_time_range = [-1.0, 5] # decision前后  0.9
baseline_range = [-1.1, 1.0]  # baseline前后 0.3s

# 生成时间序列
time_range = np.arange(baseline_range[0]-baseline_range[1] + decision_time_range[0], decision_time_range[1]-0.001, 0.004)

all_sub_imminent_data = []
all_sub_moderate_data = []
imminent_trial_num, moderate_trial_num = 0, 0
for i in range(len(subjects)):

    # 导入数据 删除眼电 设置电极位置
    EEG_epochs = mne.read_epochs_eeglab(epoch_path + 'sub' + str(subjects[i]) + '.set')
    # EEG_epochs = mne.read_epochs_eeglab(epoch_path + 'sub' + str(subjects[i]) + '_1_2.set')

    EEG_epochs.drop_channels(['VEOG'])
    EEG_epochs.set_montage(mymontage)
    EEG_epochs.crop(-1.1, 16.1)
    # 降采样
    EEG_epochs.resample(250)

    # 导入行为数据
    hebavior_trial = pd.read_excel(behavior_path + str(subjects[i]) + '/subject_v3.xlsx')

    # 统一计算tfr
    tfr_all_epoch = tfr_morlet(EEG_epochs, freqs, n_cycles=n_cycles, return_itc=False, average = False, use_fft=True, n_jobs=5)

    # 基线校正
    tfr_all_epoch.apply_baseline(mode='zlogratio', baseline=(-0.5, -0.1)) # 'mean' | 'ratio' | 'logratio' | 'percent' | 'zscore' | 'zlogratio'
    

    # create a new epochs info
    info = mne.create_info(ch_names = EEG_epochs.ch_names, ch_types = 'eeg', sfreq = 250)


    tfr_all_epoch_mne = mne.EpochsArray(data = np.mean(tfr_all_epoch.data, axis=2), info = info, tmin=EEG_epochs.tmin)

    tfr_all_epoch_mne.set_montage(mymontage)
    tfr_all_epoch_mne.crop(-1, 16)
    tfr_all_epoch_mne.resample(100)

    output_path = 'E:/workspace/study2_escape_task_eeg/tfr_data/all_trial_data_zlogratio/'
    tfr_all_epoch_mne.save(output_path + str(subjects[i])+ 'tfr.fif', overwrite=True)
